ERP Analysis Procedure**

### **1. Load Raw EEG File**

* Load the `.set` EEG file for a participant using MNE.

---

### **2. Re-reference**

* Re-reference EEG data to the **average of LM (Left Mastoid Channel 57
)** and **RM (Right Mastoid Channel 100)** **before any other step**.

---

### **3. ICA for Blink Artifact Removal**

* Run **ICA on the unfiltered or minimally filtered data** (preferably **raw or lightly filtered**, e.g., 0.01–50 Hz).
* Use **all 128 channels** (not just 81).
* **Plot components** to visually inspect and **identify blink-related components**.

  * These are typically **among the first 5** due to high amplitude.
* **Manually exclude blink components only** (avoid over-correction).

---

### **4. Apply ICA**

* Apply ICA solution to the raw data.
* Visually **inspect ICA-corrected signal** (before filtering).

---

### **5. Interpolate Bad Channels**

* Identify bad/noisy channels (especially **central anterior/medial** regions).
* **Interpolate them using weighted interpolation**.
* Try to limit exclusions; preserve as much spatial coverage as possible.

---

### **6. Filter**

* Apply band-pass filter **from 0.01 Hz to 50 Hz**.

  * Can be done before or after ICA depending on blink removal performance.
* Apply **notch filter at 60 Hz** to remove line noise.
# New section
---

### **7. Extract Events**

* Extract events from annotations:

  * `Wd2E` = normal context
  * `Wd2N` = slowed context

---

### **8. Create Epochs**

* Epochs should be **time-locked to event markers** (`Wd2E`/`Wd2N`).
* Use **epoch window**: -100 ms to 800 ms.
* Apply **baseline correction** using the **100 ms pre-onset period**.

---

### **9. Reject Artifactual Epochs**

* Automatically or manually reject epochs with high-amplitude noise or eye movement remnants.
* Use voltage thresholds, peak-to-peak rejection, or visual inspection.

---

### **10. Save Cleaned Epochs**

* Save the cleaned and processed epochs per subject (`.fif` format).

---

### **11. Compute Subject-level ERPs**

* For each subject, compute average ERPs:

  * ERP for all `Wd2E` epochs (normal context)
  * ERP for all `Wd2N` epochs (slowed context)

---

### **12. Grand Average (Group-Level ERPs)**

* Average across all 21 participants:

  * Grand average ERP for `Wd2E`
  * Grand average ERP for `Wd2N`

---

### **13. Plot ERPs (9 Scalp Regions)**

* Use all **128 channels** and group into 9 regions:

  * Left/Medial/Right × Anterior/Central/Posterior
* Plot ERP waveforms over time (−100 ms to 400 ms) showing both conditions.
* Highlight **N100** and **N280** components.




In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
import mne
import os
import gc
import numpy as np
from scipy.signal import hilbert
import matplotlib.pyplot as plt

participants = ['3', '6', '7', '8', '9', '10', '11', '12', '13', '14',
                '15', '16', '17', '18', '19', '20', '21', '22', '23',
                '24', '25', '26', '27']


In [ ]:
import os
import mne
import gc

# Set your EEG data folder path
root_dir = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData'

def preprocess_and_save_with_ica(participant):
    participant_path = os.path.join(root_dir, participant)
    set_files = [f for f in os.listdir(participant_path) if f.endswith('.set')]

    if not set_files:
        print(f"No .set files found for {participant}")
        return None

    for set_file in set_files:
        set_name = os.path.splitext(set_file)[0]
        raw_path = os.path.join(participant_path, set_file)
        fif_path = os.path.join(participant_path, f'{set_name}_clean_raw.fif')
        ica_path = os.path.join(participant_path, f'{set_name}-ica.fif')

        if os.path.exists(fif_path) and os.path.exists(ica_path):
            print(f"Already preprocessed: {set_file}")
            continue

        try:
            raw = mne.io.read_raw_eeglab(raw_path, preload=True)

            # Apply montage (necessary for interpolation to work properly)
            montage = mne.channels.make_standard_montage('GSN-HydroCel-128')
            raw.set_montage(montage)

            # Re-reference to LM/RM (E57, E100)
            try:
                lm = raw.copy().pick(['E57']).get_data()
                rm = raw.copy().pick(['E100']).get_data()
                mastoid_avg = (lm + rm) / 2
                raw._data = raw.get_data() - mastoid_avg
                print(f"{set_file}: Re-referenced using LM/RM average.")
            except Exception as e:
                print(f"{set_file}: Error during referencing — {e}")
                continue

            ### INSERTED BAD CHANNEL DETECTION & INTERPOLATION HERE ###

            # (OPTION 1: hardcoded bad channels)
            raw.info['bads'] = []  # <-- Insert your bad channels list here

            # (OPTION 2: very crude amplitude-based detection)
            # Pick EEG channels only
            picks = mne.pick_types(raw.info, eeg=True, exclude=[])
            data = raw.get_data(picks)
            ch_means = data.std(axis=1)
            threshold = 5 * ch_means.mean()
            bad_channels = [raw.ch_names[p] for p, val in enumerate(ch_means) if val > threshold]
            raw.info['bads'] = bad_channels
            print(f"Detected bad channels: {raw.info['bads']}")

            # Interpolate bad channels
            raw.interpolate_bads(reset_bads=True)
            print(f"Interpolated bad channels.")

            ### CONTINUE WITH YOUR ORIGINAL PIPELINE ###

            # ICA preparation
            raw_for_ica = raw.copy().filter(1., None)
            raw_for_ica.resample(125, npad='auto')
            raw_for_ica.crop(tmin=0, tmax=600)
            picks = mne.pick_types(raw_for_ica.info, eeg=True, exclude='bads')
            raw_for_ica.pick(picks)

            # Fit ICA
            ica = mne.preprocessing.ICA(n_components=0.99, method='fastica', random_state=97, max_iter=500)
            ica.fit(raw_for_ica)
            print(f"{set_file}: ICA fitted.")

            # Save ICA solution
            ica.save(ica_path, overwrite=True)
            print(f"{set_file}: ICA saved.")

            # Plot & save ICA components (safe saving loop)
            figs = ica.plot_components(inst=raw, show=False)
            for i, fig in enumerate(figs):
                fig_path = os.path.join(participant_path, f"{set_name}_ica_components_page_{i+1}.png")
                fig.savefig(fig_path, dpi=150)

            # Plot & save ICA sources
            fig_sources = ica.plot_sources(raw, show=False)
            if isinstance(fig_sources, list):
                for i, fig in enumerate(fig_sources):
                    fig_path = os.path.join(participant_path, f"{set_name}_ica_sources_page_{i+1}.png")
                    fig.savefig(fig_path, dpi=150)
            else:
                fig_sources_path = os.path.join(participant_path, f"{set_name}_ica_sources.png")
                fig_sources.savefig(fig_sources_path, dpi=150)

            # Save re-referenced raw data
            raw.save(fif_path, overwrite=True)

            # Free memory after each file
            del raw, raw_for_ica, ica, figs, fig_sources
            gc.collect()

        except Exception as e:
            print(f"{set_file}: Error during processing — {e}")
            continue


In [ ]:
preprocess_and_save_with_ica('3')

In [ ]:
preprocess_and_save_with_ica('6')

In [ ]:
preprocess_and_save_with_ica('7')

In [ ]:

preprocess_and_save_with_ica('8')



In [ ]:
preprocess_and_save_with_ica('10')


In [ ]:
preprocess_and_save_with_ica('11')



In [ ]:
preprocess_and_save_with_ica('12')


In [ ]:
preprocess_and_save_with_ica('13')


In [ ]:
preprocess_and_save_with_ica('14')


In [ ]:
preprocess_and_save_with_ica('15')


In [ ]:
preprocess_and_save_with_ica('16')


In [ ]:
preprocess_and_save_with_ica('17')


In [ ]:
preprocess_and_save_with_ica('18')


In [ ]:
preprocess_and_save_with_ica('19')


In [ ]:
preprocess_and_save_with_ica('20')


In [ ]:
preprocess_and_save_with_ica('21')


In [ ]:
preprocess_and_save_with_ica('22')


In [ ]:
preprocess_and_save_with_ica('24')


In [ ]:
preprocess_and_save_with_ica('25')


In [ ]:
preprocess_and_save_with_ica('26')


In [ ]:
preprocess_and_save_with_ica('27')

In [ ]:
import os
import mne

# EEG data folder
root_dir = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData'

def apply_ica_artifact_removal(participant):
    participant_path = os.path.join(root_dir, participant)
    fif_files = [f for f in os.listdir(participant_path) if f.endswith('_clean_raw.fif')]

    if not fif_files:
        print(f"No cleaned raw file found for participant {participant}")
        return None

    for fif_file in fif_files:
        set_name = fif_file.replace('_clean_raw.fif', '')
        fif_path = os.path.join(participant_path, fif_file)
        ica_path = os.path.join(participant_path, f"{set_name}-ica.fif")

        # Load raw and ICA
        raw = mne.io.read_raw_fif(fif_path, preload=True)
        ica = mne.preprocessing.read_ica(ica_path)

        #### Automatic EOG-based artifact rejection ####

        # Use frontal electrodes as EOG proxies
        eog_inds, eog_scores = ica.find_bads_eog(raw, ch_name=['E1', 'E17', 'E128', 'E8'])
        print(f"{set_name}: EOG components detected: {eog_inds}")

        # No ECG in your dataset
        ecg_inds = []

        # Combine components to exclude
        ica.exclude = list(set(eog_inds + ecg_inds))
        print(f"{set_name}: ICA components marked for exclusion: {ica.exclude}")

        # Apply ICA artifact correction
        raw_clean = ica.apply(raw.copy())
        print(f"{set_name}: ICA applied, artifacts removed.")

        # Save final fully cleaned file
        final_clean_path = os.path.join(participant_path, f"{set_name}_ica_applied_raw.fif")
        raw_clean.save(final_clean_path, overwrite=True)
        print(f"{set_name}: Fully cleaned file saved: {final_clean_path}")

        # Clean up memory
        del raw, ica, raw_clean



In [ ]:
apply_ica_artifact_removal('3')


| File                  | Components removed |
| --------------------- | ------------------ |
| `3_clean_raw.fif`     | None               |
| `3_001_clean_raw.fif` | 0, 9, 55           |
| `3_002_clean_raw.fif` | 0, 9, 55           |


In [ ]:
apply_ica_artifact_removal('6')

| File                  | EOG components detected | ICA components excluded |
| --------------------- | ----------------------- | ----------------------- |
| `6_clean_raw.fif`     | `[2]`                   | `[2]`                   |
| `6_001_clean_raw.fif` | `[5, 3, 51, 41, 63]`    | `[3, 5, 41, 51, 63]`    |
| `6_002_clean_raw.fif` | `[5, 3, 51, 41, 63]`    | `[3, 5, 41, 51, 63]`    |


In [ ]:
apply_ica_artifact_removal('7')

| File                  | ICA Components Detected (EOG) | Components Removed |
| --------------------- | ----------------------------- | ------------------ |
| `7_clean_raw.fif`     | `[]`                          | none               |
| `7_001_clean_raw.fif` | `[51, 44, 57, 8]`             | `[8, 57, 51, 44]`  |
| `7_002_clean_raw.fif` | `[51, 44, 57, 8]`             | `[8, 57, 51, 44]`  |


In [ ]:
apply_ica_artifact_removal('8')

| File                  | ICA Components Detected (EOG) | Components Removed |
| --------------------- | ----------------------------- | ------------------ |
| `8_clean_raw.fif`     | `[0]`                         | `[0]`              |
| `8_001_clean_raw.fif` | `[3, 0]`                      | `[0, 3]`           |
| `8_002_clean_raw.fif` | `[3, 0]`                      | `[0, 3]`           |


In [ ]:
apply_ica_artifact_removal('10')

| File                   | EOG components detected                                 | ICA components removed |
| ---------------------- | ------------------------------------------------------- | ---------------------- |
| `10_clean_raw.fif`     | none                                                    | none                   |
| `10_001_clean_raw.fif` | \[29, 26, 61, 36, 17]                                   | 5 components           |
| `10_002_clean_raw.fif` | \[29, 26, 61, 36, 17]                                   | 5 components           |
| `10_003_clean_raw.fif` | \[16, 18, 75, 52, 17, 23, 10, 1, 7, 30, 5, 51, 6, 4, 8] | 15 components          |


In [ ]:
apply_ica_artifact_removal('11')

| File                   | EOG components detected | ICA components removed |
| ---------------------- | ----------------------- | ---------------------- |
| `11_clean_raw.fif`     | none                    | none                   |
| `11_001_clean_raw.fif` | 11 components           | 11 components removed  |
| `11_002_clean_raw.fif` | 11 components           | 11 components removed  |


In [ ]:
apply_ica_artifact_removal('12')

| File                   | EOG components detected | ICA components removed |
| ---------------------- | ----------------------- | ---------------------- |
| `12_clean_raw.fif`     | none                    | none                   |
| `12_001_clean_raw.fif` | 8 components            | 8 components removed   |
| `12_002_clean_raw.fif` | 8 components            | 8 components removed   |


In [ ]:
apply_ica_artifact_removal('13')

In [ ]:
apply_ica_artifact_removal('14')

In [ ]:
apply_ica_artifact_removal('15')

In [ ]:
apply_ica_artifact_removal('16')


In [ ]:
apply_ica_artifact_removal('17')

In [ ]:
apply_ica_artifact_removal('18')

In [ ]:
apply_ica_artifact_removal('19')

In [ ]:
apply_ica_artifact_removal('20')

In [ ]:
apply_ica_artifact_removal('21')

In [ ]:
apply_ica_artifact_removal('22')

In [ ]:
apply_ica_artifact_removal('24')

In [ ]:
apply_ica_artifact_removal('25')

In [ ]:
apply_ica_artifact_removal('26')

In [ ]:
apply_ica_artifact_removal('27')